In [8]:
import time

import pandas as pd

import math

import numpy as np

start = time.time()

df_estruturas = pd.read_excel(r"C:\Users\Usuario\Desktop\Pessoal\Python\Separador de Strings\Estruturas Riodades.xlsx")
df_inversores = pd.read_excel(r"C:\Users\Usuario\Desktop\Pessoal\Python\Separador de Strings\Inversores Riodades.xlsx")

df_estruturas['TAG'] = df_estruturas['PST'].astype(str).str.zfill(2) + '.' + df_estruturas['INV'].astype(str).str.zfill(2)
df_inversores['TAG'] = df_inversores['PST'].astype(str).str.zfill(2) + '.' + df_inversores['INV'].astype(str).str.zfill(2)

df_estruturas['Comparing X'] = df_estruturas.groupby('TAG')['Position X'].transform('mean')

df_estruturas = df_estruturas.merge(df_inversores, on='TAG')

df_estruturas['E or W'] = np.where(
    df_estruturas['Position X_y'] >= df_estruturas['Comparing X'],
    'E',
    'W'
)

print(df_estruturas)

      Position X_x  Position Y_x EF  INV_x                 Layer_x  PST_x  \
0        54597.240    150598.353  -      5      EMF_UFV_Riodades_1      1   
1        54597.240    150589.353  -      5      EMF_UFV_Riodades_1      1   
2        54597.240    150580.353  -      5      EMF_UFV_Riodades_1      1   
3        54597.240    150571.353  -      5      EMF_UFV_Riodades_1      1   
4        54597.240    150535.353  -      7      EMF_UFV_Riodades_1      1   
...            ...           ... ..    ...                     ...    ...   
2853     55717.518    149821.100  -      7  EMF_UFV_Riodades_8_705      8   
2854     55717.518    149866.100  -      7  EMF_UFV_Riodades_8_705      8   
2855     55717.518    149830.100  -      7  EMF_UFV_Riodades_8_705      8   
2856     55717.518    149839.100  -      7  EMF_UFV_Riodades_8_705      8   
2857     55717.518    149857.100  -      7  EMF_UFV_Riodades_8_705      8   

     UFV_x    TAG  Comparing X  DSJ  GRP  INV_y                 Layer_y  \


In [9]:
def group_structures(df, max_distance=20):
    """
    Groups adjacent structures in pairs (2) while respecting:
    - TAG boundaries
    - Maximum distance constraint (20m default)
    - Sorted by Position Y then X
    """
    # Sort by Position X (primary) and Position Y (secondary)
    df = df.sort_values(['Position Y_x', 'Position X_x'], ascending=[False, True]).reset_index(drop=True)

    # Calculate pairwise distances between consecutive structures
    df['x_diff'] = df['Position X_x'].diff()
    df['y_diff'] = df['Position Y_x'].diff()
    df['distance'] = np.sqrt(df['x_diff']**2 + df['y_diff']**2)

    # Create group boundaries where:
    # 1. TAG changes OR
    # 2. Distance > max_distance OR
    # 3. It's the first structure
    df['group_boundary'] = (
        (df['TAG'] != df['TAG'].shift(1)) |
        (df['distance'] > max_distance) |
        (df.index == 0)
    )

    # Create group IDs
    df['group_id'] = df['group_boundary'].cumsum()

    # Split into groups
    result_groups = []
    for _, group_df in df.groupby('group_id'):
        # Split groups larger than 2 into pairs
        size = len(group_df)
        if size <= 2:
            result_groups.append(group_df)
            continue

        n_pairs = size // 2
        for i in range(n_pairs):
            result_groups.append(group_df.iloc[i*2:i*2+2])

        if size % 2:  # Handle odd-sized groups
            result_groups.append(group_df.iloc[-1:])

    return result_groups

def create_group_positions(groups):
    """Create the output DataFrame from groups (same as before)"""
    group_data = []
    for i, group_df in enumerate(groups, 1):
        rep_struct = group_df.iloc[0]
        group_data.append({
            'Group_ID': i,
            'TAG': rep_struct['TAG'],
            'Position_X': rep_struct['Position X_x'],
            'Position_Y': rep_struct['Position Y_x'],
            'Group_Size': len(group_df),
            'Layer': rep_struct['Layer_x']
        })
    return pd.DataFrame(group_data)

results_df = group_structures(df_estruturas)
group_positions = create_group_positions(results_df)
group_positions["PST"] = group_positions['TAG'].str[1]
group_positions["INV"] = group_positions['TAG'].str[4]
# For row-wise operation (recommended)
group_positions['Script'] = group_positions.apply(
    lambda row: (
        f'(command-s "._layer" "set" "{row["Layer"]}" "") '
        f'(command-s "._color" "ByLayer" "") '
        f'(command-s "._insert" "EMF309_estrutura_fixa_{row["Group_Size"]}" '
        f'"{row["Position_X"]},{row["Position_Y"]}" 1 1 0 "-" {row["INV"].zfill(2)} {row["PST"].zfill(2)} "-")'
    ),
    axis=1
)

print(group_positions)

with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Separador de Strings\Separador de Strings.scr", 'w') as f:
    f.write('\n'.join(group_positions["Script"].astype(str)) + '\n')

      Group_ID    TAG  Position_X  Position_Y  Group_Size  \
0            1  03.01   55462.190  150776.472           2   
1            2  03.01   55501.840  150776.472           2   
2            3  03.01   55541.490  150776.472           2   
3            4  03.01   55462.190  150767.472           2   
4            5  03.01   55501.840  150767.472           2   
...        ...    ...         ...         ...         ...   
1531      1532  08.20   55526.276  149595.100           2   
1532      1533  08.20   55565.966  149595.100           2   
1533      1534  08.20   55605.656  149595.100           2   
1534      1535  08.19   55421.051  149586.100           2   
1535      1536  08.19   55460.741  149586.100           2   

                       Layer PST INV  \
0         EMF_UFV_Riodades_3   3   1   
1         EMF_UFV_Riodades_3   3   1   
2         EMF_UFV_Riodades_3   3   1   
3         EMF_UFV_Riodades_3   3   1   
4         EMF_UFV_Riodades_3   3   1   
...                      ..